# FIVAR 모델 평가 및 포트폴리오 분석

이 노트북은 'Factor and Idiosyncratic VAR Volatility Matrix Models for Heavy-Tailed High-Frequency Financial Observations' 논문에 기반하여 FIVAR 모델을 구현하고 평가합니다.

## 1. 라이브러리 임포트 및 기본 설정

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.linalg import eigh, inv
from numpy.linalg import norm
from sklearn.linear_model import HuberRegressor, Lasso, LinearRegression
from sklearn.exceptions import ConvergenceWarning
import warnings
from tqdm.notebook import tqdm  # 주피터 노트북 환경에 맞는 tqdm 임포트
from typing import Dict, List, Tuple
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import seaborn as sns

# --- 경고 메시지 무시 설정 ---
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

## 2. 데이터 로딩 및 전처리 헬퍼 함수

In [ ]:
def load_csv_data(file_path: str) -> Tuple[np.ndarray, pd.DatetimeIndex, List[str]]:
    """Long-format의 CSV 파일에서 PRVM 데이터를 불러와 3D 텐서로 변환합니다."""
    print(f"'{file_path}' 파일에서 PRVM 데이터를 불러옵니다...")
    try:
        df = pd.read_csv(file_path, parse_dates=['date'])
        unique_dates = sorted(df['date'].unique())
        tickers = sorted(pd.unique(df[['ticker_i', 'ticker_j']].values.ravel('K')))
        num_days = len(unique_dates)
        num_assets = len(tickers)
        
        print(f"총 {num_days}일, {num_assets}개의 자산 데이터를 처리합니다.")

        prvm_data = np.zeros((num_days, num_assets, num_assets))
        ticker_map = {ticker: i for i, ticker in enumerate(tickers)}
        
        df['i'] = df['ticker_i'].map(ticker_map)
        df['j'] = df['ticker_j'].map(ticker_map)

        for t, date in enumerate(tqdm(unique_dates, desc="데이터 텐서 변환 중")):
            daily_data = df[df['date'] == date]
            matrix = np.zeros((num_assets, num_assets))
            matrix[daily_data['i'], daily_data['j']] = daily_data['value']
            matrix[daily_data['j'], daily_data['i']] = daily_data['value']
            prvm_data[t] = matrix

    except Exception as e:
        print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
        return None, None, None

    dates = pd.to_datetime(unique_dates)
    print(f"데이터 로딩 및 변환 완료. Shape: {prvm_data.shape}")
    print(f"날짜 범위: {dates.min().date()} ~ {dates.max().date()}")
    return prvm_data, dates, tickers

def load_gics_data(gics_file_path: str, tickers: List[str]) -> np.ndarray:
    """GICS 분류가 담긴 CSV 파일을 불러옵니다."""
    print(f"'{gics_file_path}' 파일에서 GICS 데이터를 불러옵니다...")
    try:
        gics_df = pd.read_csv(gics_file_path, header=None)
        gics_map = pd.Series(gics_df.iloc[1:, 1].values, index=gics_df.iloc[1:, 0].values)
        gics_sectors = gics_map.reindex(tickers).fillna('Unknown').to_numpy()
        if len(gics_sectors) != len(tickers):
            raise ValueError(f"GICS 데이터의 개수({len(gics_sectors)})와 자산의 수({len(tickers)})가 일치하지 않습니다.")
        return gics_sectors
    except Exception as e:
        raise IOError(f"GICS 파일을 읽는 중 오류가 발생했습니다: {e}")

def project_psd(matrix: np.ndarray) -> np.ndarray:
    """행렬을 양의 준정부호(Positive Semi-Definite) 행렬로 변환합니다."""
    eigenvalues, eigenvectors = eigh(matrix)
    eigenvalues[eigenvalues < 1e-10] = 1e-10
    psd_matrix = eigenvectors @ np.diag(eigenvalues) @ eigenvectors.T
    return (psd_matrix + psd_matrix.T) / 2

## 3. 성능 평가 함수 (MSPE, QLIKE)

In [ ]:
def calculate_mspe(forecasts: List[np.ndarray], ground_truths: List[np.ndarray]) -> float:
    """MSPE (Mean Squared Prediction Error)를 계산합니다."""
    if not forecasts: return np.nan
    errors = [norm(forecasts[i] - ground_truths[i], 'fro')**2 for i in range(len(forecasts))]
    return np.mean(errors)

def calculate_qlike(forecasts: List[np.ndarray], ground_truths: List[np.ndarray]) -> float:
    """QLIKE 손실 함수를 계산합니다."""
    if not forecasts: return np.nan
    qlike_vals = []
    for i in range(len(forecasts)):
        try:
            forecast_psd = project_psd(forecasts[i])
            forecast_reg = forecast_psd + np.eye(forecast_psd.shape[0]) * 1e-10
            sign, log_det_val = np.linalg.slogdet(forecast_reg)
            if sign <= 0: continue
            trace_val = np.trace(inv(forecast_reg) @ ground_truths[i])
            qlike_vals.append(log_det_val + trace_val)
        except np.linalg.LinAlgError:
            continue
    return np.mean(qlike_vals) if qlike_vals else np.nan

## 4. 예측 모델 구현

In [ ]:
def calculate_poet_prvm(prvm_matrix: np.ndarray, gics_sectors: np.ndarray, num_factors: int) -> np.ndarray:
    """벤치마크 모델: 단일 PRVM 행렬로부터 POET-PRVM을 계산합니다."""
    p = prvm_matrix.shape[0]
    prvm_psd = project_psd(prvm_matrix)
    
    eigenvalues, eigenvectors = eigh(prvm_psd)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues, eigenvectors = eigenvalues[idx], eigenvectors[:, idx]
    
    factor_part = np.zeros_like(prvm_psd)
    for i in range(num_factors):
        factor_part += eigenvalues[i] * np.outer(eigenvectors[:, i], eigenvectors[:, i])
        
    idiosyncratic_part_raw = prvm_psd - factor_part
    
    same_sector_mask = (gics_sectors.reshape(-1, 1) == gics_sectors)
    np.fill_diagonal(same_sector_mask, True)
    idiosyncratic_part_thresholded = idiosyncratic_part_raw * same_sector_mask
    
    poet_prvm = factor_part + idiosyncratic_part_thresholded
    return project_psd(poet_prvm)

def get_idio_matrices(prvm_data: np.ndarray, gics_sectors: np.ndarray, r: int) -> np.ndarray:
    """FIVAR 모델 전처리: POET 기법으로 특이 변동성 행렬을 계산합니다."""
    print("POET 기법을 적용하여 특이 변동성 행렬을 생성합니다 (병렬 처리)...")
    p = prvm_data.shape[1]
    same_sector_mask = (gics_sectors[:, None] == gics_sectors)
    np.fill_diagonal(same_sector_mask, True)
    
    def process_day(prvm_matrix):
        prvm_psd = project_psd(prvm_matrix)
        eigenvalues, eigenvectors = eigh(prvm_psd)
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues, eigenvectors = eigenvalues[idx], eigenvectors[:, idx]
        
        factor_part = np.zeros_like(prvm_psd)
        for i in range(r):
            factor_part += eigenvalues[i] * np.outer(eigenvectors[:, i], eigenvectors[:, i])
            
        idiosyncratic_part_raw = prvm_psd - factor_part
        return idiosyncratic_part_raw * same_sector_mask

    idio_matrices = Parallel(n_jobs=-1)(delayed(process_day)(prvm) for prvm in tqdm(prvm_data, desc="POET 적용 중"))
    return np.array(idio_matrices)

def fit_idio_component(args: Tuple) -> Tuple[int, HuberRegressor, float]:
    """FIVAR 모델 헬퍼: BIC를 사용하여 최적의 c_eta를 찾고 모델을 학습합니다."""
    i, y_i, X, p, n_reg, c_eta_candidates, model_type = args
    best_bic_i, best_c_eta_i = np.inf, None

    if model_type in ['h-lasso', 'lasso']:
        for c_eta in c_eta_candidates:
            eta_I = c_eta * np.sqrt(np.log(p) / n_reg)
            
            if model_type == 'h-lasso':
                reg = HuberRegressor(fit_intercept=True, alpha=eta_I, epsilon=1.345)
            else:
                reg = Lasso(alpha=eta_I, fit_intercept=True, max_iter=2000)

            reg.fit(X, y_i)
            y_pred = reg.predict(X)
            rss = np.sum((y_i - y_pred)**2)
            num_params = np.sum(np.abs(reg.coef_) > 1e-6) + (1 if reg.intercept_ != 0 else 0)
            
            bic = n_reg * np.log(rss / n_reg) + num_params * np.log(n_reg) if rss > 0 else np.inf
            if bic < best_bic_i:
                best_bic_i, best_c_eta_i = bic, c_eta
        
        final_eta_I = best_c_eta_i * np.sqrt(np.log(p) / n_reg)
    else: # OLS
        final_eta_I = 0.0
        best_c_eta_i = 0.0

    if model_type == 'h-lasso':
        final_regressor = HuberRegressor(fit_intercept=True, alpha=final_eta_I)
    elif model_type == 'lasso':
        final_regressor = Lasso(alpha=final_eta_I, fit_intercept=True, max_iter=2000)
    else: # OLS
        final_regressor = LinearRegression(fit_intercept=True)

    final_regressor.fit(X, y_i)
    
    return i, final_regressor, best_c_eta_i

class FIVARModel:
    """FIVAR 모델의 학습 및 예측을 담당하는 메인 클래스."""
    def __init__(self, r: int, model_type: str, h: int = 1, l: int = 22):
        self.r, self.h, self.l = r, h, l
        self.model_type = model_type.lower()
        self.p, self.tickers = None, None
        self.factor_eigenvectors, self.idio_eigenvectors = None, None
        self.factor_regressors, self.idio_regressors = None, None
        self.last_known_eigenvalues = None
        self.idio_mean_forecast = None # OLS 모델용

    def fit(self, prvm_matrices: Dict, idio_matrices: Dict, tickers: List[str]):
        self.tickers, self.p = tickers, len(tickers)
        all_dates = sorted(prvm_matrices.keys())
        
        if len(all_dates) < self.l:
             raise ValueError(f"고유벡터 계산에 필요한 데이터가 부족합니다 (필요: {self.l}, 보유: {len(all_dates)}).")

        last_l_dates = all_dates[-self.l:]
        avg_prvm_matrix = np.mean([prvm_matrices[d] for d in last_l_dates], axis=0)
        _, factor_evecs_all = eigh(avg_prvm_matrix)
        self.factor_eigenvectors = factor_evecs_all[:, -self.r:][:, ::-1]

        avg_idio_matrix = np.mean([idio_matrices[d] for d in last_l_dates], axis=0)
        _, idio_evecs_all = eigh(avg_idio_matrix)
        self.idio_eigenvectors = idio_evecs_all[:, ::-1]

        factor_evals = [np.diag(self.factor_eigenvectors.T @ prvm_matrices[d] @ self.factor_eigenvectors) / self.p for d in all_dates]
        idio_evals = [np.diag(self.idio_eigenvectors.T @ idio_matrices[d] @ self.idio_eigenvectors) for d in all_dates]
        
        eigenvalue_df = pd.DataFrame([np.concatenate(z) for z in zip(factor_evals, idio_evals)], index=all_dates)

        y = eigenvalue_df.iloc[self.h:]
        X_list = [eigenvalue_df.shift(k).iloc[self.h:] for k in range(1, self.h + 1)]
        X = pd.concat(X_list, axis=1)
        
        n_reg = len(y)
        
        self.factor_regressors = [None] * self.r
        X_factor = X.iloc[:, :self.r * self.h]
        for i in range(self.r):
            if self.model_type == 'h-lasso':
                self.factor_regressors[i] = HuberRegressor(fit_intercept=True, alpha=0.0)
            else:
                self.factor_regressors[i] = LinearRegression(fit_intercept=True)
            self.factor_regressors[i].fit(X_factor, y.iloc[:, i])

        self.idio_regressors = [None] * self.p
        if self.model_type == 'ols':
            self.idio_mean_forecast = eigenvalue_df.iloc[-self.l:, self.r:].mean().to_numpy()
        else:
            c_eta_candidates = np.linspace(0.1, 10.0, 20)
            tasks = [(i, y.iloc[:, self.r + i].copy(), X.copy(), self.p, n_reg, c_eta_candidates, self.model_type) for i in range(self.p)]
            results = Parallel(n_jobs=-1)(delayed(fit_idio_component)(task) for task in tqdm(tasks, desc=f"  {self.model_type.upper()} 계수 추정(병렬)", leave=False))
            for i, regressor, _ in results:
                self.idio_regressors[i] = regressor

        self.last_known_eigenvalues = eigenvalue_df.iloc[-self.h:].to_numpy()

    def predict(self) -> np.ndarray:
        if self.last_known_eigenvalues is None: raise RuntimeError("모델이 학습되지 않았습니다.")
        
        X_pred = self.last_known_eigenvalues.flatten().reshape(1, -1)
        pred_evals = np.zeros(self.p + self.r)
        
        X_pred_factor = X_pred[:, :self.r * self.h]
        for i in range(self.r):
            pred_evals[i] = self.factor_regressors[i].predict(X_pred_factor)[0]
            
        if self.model_type == 'ols':
            pred_evals[self.r:] = self.idio_mean_forecast
        else:
            for i in range(self.p):
                pred_evals[self.r + i] = self.idio_regressors[i].predict(X_pred)[0]
        
        pred_evals[pred_evals < 1e-10] = 1e-10
        
        factor_forecast = self.p * (self.factor_eigenvectors @ np.diag(pred_evals[:self.r]) @ self.factor_eigenvectors.T)
        idio_forecast = self.idio_eigenvectors @ np.diag(pred_evals[self.r:]) @ self.idio_eigenvectors.T
        
        return project_psd(factor_forecast + idio_forecast)

## 5. 메인 실행 로직: 롤링 윈도우 예측 및 평가

In [ ]:
# --- 설정값 ---
prvm_file_path = 'prvm_0731_final_corrected.csv' 
gics_file_path = 'gicslist.csv'
num_factors = 3
in_sample_size = 251 # 1년치 영업일
h_lag = 1 # BIC로 h=1이 선택되었다고 가정 (논문 5.2절)
l_window = 22 # 고유벡터 추정을 위한 윈도우 크기 (논문 5.2절)

# --- 데이터 로딩 ---
prvm_data_all, dates, tickers = load_csv_data(prvm_file_path)
if prvm_data_all is not None:
    gics_data = load_gics_data(gics_file_path, tickers)

    # --- 특이 변동성 행렬 사전 계산 ---
    idio_data_all = get_idio_matrices(prvm_data_all, gics_data, num_factors)

    # --- 롤링 윈도우 예측 ---
    periods = {
        "Period 1 (2018-2019)": ('2018-01-01', '2019-12-31'),
        "Period 2 (2018)": ('2018-01-01', '2018-12-31'),
        "Period 3 (2019)": ('2019-01-01', '2019-12-31'),
    }
    
    models = ['POET-PRVM', 'OLS', 'LASSO', 'H-LASSO']
    forecasts = {model: [] for model in models}
    ground_truths, forecast_dates = [], []

    start_idx = dates.searchsorted(pd.to_datetime('2018-01-01'))
    if start_idx < in_sample_size:
        start_idx = in_sample_size
        print(f"데이터가 충분하지 않아 {dates[start_idx].date()}부터 예측을 시작합니다.")

    for t in tqdm(range(start_idx, len(dates)), desc="전체 롤링 윈도우 예측"):
        # In-sample 데이터셋 정의
        train_start_idx = t - in_sample_size
        train_end_idx = t
        in_sample_prvm = prvm_data_all[train_start_idx:train_end_idx]
        in_sample_idio = idio_data_all[train_start_idx:train_end_idx]
        in_sample_dates = dates[train_start_idx:train_end_idx]
        
        prvm_train_dict = {d: m for d, m in zip(in_sample_dates, in_sample_prvm)}
        idio_train_dict = {d: m for d, m in zip(in_sample_dates, in_sample_idio)}
        
        # Ground truth (실현 변동성) 정의
        ground_truth_matrix = prvm_data_all[t]
        ground_truths.append(calculate_poet_prvm(ground_truth_matrix, gics_data, num_factors))
        forecast_dates.append(dates[t])

        # --- 모델별 예측 수행 ---
        # 1. POET-PRVM (전일 값 사용)
        prvm_t_minus_1 = in_sample_prvm[-1]
        forecasts['POET-PRVM'].append(calculate_poet_prvm(prvm_t_minus_1, gics_data, num_factors))
        
        # 2. FIVAR 모델들 (OLS, LASSO, H-LASSO)
        for model_type in ['OLS', 'LASSO', 'H-LASSO']:
            model = FIVARModel(r=num_factors, model_type=model_type, h=h_lag, l=l_window)
            model.fit(prvm_train_dict, idio_train_dict, tickers)
            prediction = model.predict()
            forecasts[model_type].append(prediction)

## 6. 결과 집계 및 출력

In [ ]:
print("\n--- 최종 평가 결과 (논문 Table 2 형식) ---")
final_results = []
forecast_dates_pd = pd.to_datetime(forecast_dates)

for period_name, (start_date, end_date) in periods.items():
    period_mask = (forecast_dates_pd >= start_date) & (forecast_dates_pd <= end_date)
    period_truths = [ground_truths[i] for i, val in enumerate(period_mask) if val]
    
    for model in models:
        period_forecasts = [forecasts[model][i] for i, val in enumerate(period_mask) if val]
        mspe_val = calculate_mspe(period_forecasts, period_truths)
        qlike_val = calculate_qlike(period_forecasts, period_truths)
        
        final_results.append({
            "Period": period_name, "Model": model,
            "MSPE (x10^4)": mspe_val * 10**4,
            "QLIKE x 10^-3": qlike_val * 10**-3,
        })

results_df = pd.DataFrame(final_results).pivot_table(index='Model', columns='Period', values=['MSPE (x10^4)', 'QLIKE x 10^-3'])

# 표 형식으로 재구성하여 출력
model_order = ['POET-PRVM', 'OLS', 'LASSO', 'H-LASSO']
summary_table = results_df.swaplevel(0, 1, axis=1).sort_index(axis=1)
display(summary_table.loc[model_order].style.format("{:.3f}"))

## 7. 포트폴리오 리스크 분석 (주석 처리됨)

아래 셀은 실제 일별 수익률 데이터(`daily_returns.csv`)가 있을 경우, 주석을 해제하여 포트폴리오 리스크를 분석하고 시각화하는 데 사용할 수 있습니다.

In [ ]:
# print("\n--- 포트폴리오 리스크 분석 (논문 Figure 5) ---")
# 
# try:
#     # 실제 분석을 위해서는 아래 주석을 해제하고 'daily_returns.csv' 파일 경로를 지정해야 합니다.
#     # daily_returns_df = pd.read_csv('daily_returns.csv', index_col='date', parse_dates=True)
#     # daily_returns_df = daily_returns_df[tickers].loc[forecast_dates_pd]
#     
#     plt.style.use('seaborn-v0_8-whitegrid')
#     fig, axes = plt.subplots(1, 3, figsize=(21, 6), sharey=True)
#     
#     exposure_constraints = np.linspace(1.0, 3.0, 21)
# 
#     for i, (period_name, (start_date, end_date)) in enumerate(periods.items()):
#         ax = axes[i]
#         period_mask = (forecast_dates_pd >= start_date) & (forecast_dates_pd <= end_date)
#         
#         for model in models:
#             # # 실제 데이터가 있을 경우 아래 로직을 사용하여 리스크를 계산합니다.
#             # model_risks = []
#             # for c0 in tqdm(exposure_constraints, desc=f'Risk for {model} in {period_name}', leave=False):
#             #     daily_portfolio_variances = []
#             #     period_forecasts = [forecasts[model][j] for j, val in enumerate(period_mask) if val]
#             #     period_returns = daily_returns_df[period_mask]
# 
#             #     for day_idx in range(len(period_forecasts)):
#             #         Sigma_hat = period_forecasts[day_idx]
#             #         # 최적화 문제 풀이 (실제로는 제약조건을 만족하는 quadratic programming solver 사용 필요)
#             #         try:
#             #             Sigma_inv = inv(Sigma_hat)
#             #             ones = np.ones(Sigma_hat.shape[0])
#             #             w = (Sigma_inv @ ones) / (ones.T @ Sigma_inv @ ones)
#             #             ret = period_returns.iloc[day_idx].to_numpy()
#             #             portfolio_var = w.T @ (ret.reshape(-1,1) @ ret.reshape(1,-1)) @ w
#             #             daily_portfolio_variances.append(portfolio_var)
#             #         except np.linalg.LinAlgError:
#             #             continue
#             #     
#             #     if daily_portfolio_variances:
#             #         annualized_risk = np.sqrt(np.mean(daily_portfolio_variances) * 252) * 100
#             #         model_risks.append(annualized_risk)
#             #     else:
#             #         model_risks.append(np.nan)
#             # ax.plot(exposure_constraints, model_risks, marker='o', linestyle='--', label=model)

#             # 임시 데이터로 그래프 형태만 시연
#             base_risk = {'POET-PRVM': 8.5, 'OLS': 7.2, 'LASSO': 7.1, 'H-LASSO': 6.9}[model]
#             if i == 1: base_risk += 1.0
#             if i == 2: base_risk -= 0.5
#             trend = np.linspace(0, 0.5, len(exposure_constraints))
#             noise = np.random.normal(0, 0.05, len(exposure_constraints))
#             ax.plot(exposure_constraints, base_risk + trend + noise, marker='o', markersize=4, linestyle='--', label=model)

# 
#         ax.set_title(f'Portfolio Risk ({period_name.split("(")[1][:-1]})')
#         ax.set_xlabel('Exposure constraint')
#         if i == 0: ax.set_ylabel('Annualized risk (%)')
#         ax.legend()
#     
#     plt.tight_layout()
#     plt.show()
# 
# except Exception as e:
#     print(f"\n포트폴리오 리스크 분석 중 오류가 발생했습니다: {e}")
#     print("이 부분은 실제 일별 수익률 데이터가 필요하며, 현재는 시연용 데이터로 그래프를 생성합니다.")